# Part 2: Preprocessing

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/raw/train.csv')
print(df.shape)

(1460, 81)


***2.1 Identify missing and detect outliers***

-*Removing outliers*

In [4]:
df = df[~((df['GrLivArea'] > 4000)& (df['SalePrice'] < 300000))]
print(df.shape)

(1458, 81)


As a result, two rows were removed because they were identified as outliers.

-*Processing missing values*

In [5]:
df['MasVnrType'].unique() # like columns with 'NaN' values, ex:'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType',

<StringArray>
['BrkFace', nan, 'Stone', 'BrkCmn']
Length: 4, dtype: str

In [6]:
#Group 1
cols_fill_none = [  'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType',
                    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                    'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtCond']

for col in cols_fill_none:
    df[col] = df[col].fillna('None')

#Group 2
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0]) #Impute missing values using the most frequent value

-*Check the remaining missing*

In [7]:
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(remaining_missing)

Series([], dtype: int64)


***2.2 Encoding***

**The categorical features were identified and divided into two categories based on their characteristics, as different encoding methods are required.**
<i>

Category 1 – Ordinal features: These features have a clear order or ranking. They were encoded using controlled label encoding, where each category was assigned an ordered numerical value (e.g., 1, 2, 3, ...).

Category 2 – Nominal features: These features have no inherent order and simply represent different categories. They were encoded using one-hot encoding, where each category was represented by a binary variable (0 or 1).<i>

In [8]:
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']# these columns are ordinal features and will be label
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
for col in ordinal_cols:
    df[col] = df[col].map(quality_map)

df['ExterQual'].unique()#check

array([4, 3, 5, 2])

In [9]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist() # these remaining columns are nominal features and will be one-hot-encode
print(categorical_cols)
print(len(categorical_cols))

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(df.shape)

['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
33
(1458, 230)


C:\Users\Thy\AppData\Local\Temp\ipykernel_23376\2582974023.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist() # these remaining columns are nominal features and will be one-hot-encode


In [10]:
df.select_dtypes(include=['object']).columns.tolist() #check if there are any remaining categorical columns

[]

In [11]:
X = df.drop(['SalePrice'], axis=1) #axis =1 drop columns, axis = 0 drop rows
y = df['SalePrice']

print(X.shape)
print(y.shape)

(1458, 229)
(1458,)


***2.3 Data spliting(dividing data into traing and testing set)***

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)

(1166, 229)
(292, 229)


***2.4 Scaling by Standardization instead of Nomarliztion***

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

* **`scaler.fit_transform(X_train)`** performs two tasks at the same time:

  1. It **learns the scaling parameters** by calculating the **mean** and **standard deviation** of each feature in `X_train`.
  2. It **applies the learned transformation** to `X_train`, producing the standardized training data.

* **`scaler.transform(X_test)`** only **applies the scaling parameters learned from `X_train`** to `X_test`. It **does not recalculate** the mean or standard deviation using the test data.


***2.5 Saving the preprocessing objects***

In [14]:
import joblib
joblib.dump(X_train, '../data/processed/X_train.pkl')
joblib.dump(X_train_scaled, '../data/processed/X_train_scaled.pkl')
joblib.dump(X_test_scaled, '../data/processed/X_test_scaled.pkl')

joblib.dump(y_train, '../data/processed/y_train.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

joblib.dump(scaler, '../model/scaler.pkl')
joblib.dump(list(X_train.columns), '../model/feature_columns.pkl')

print('File saved successfully')

File saved successfully


## Kết luận từ Preprocessing
<b>

1. Outliers: loại 2 dòng index 523, 1298 (GrLivArea > 4000 nhưng SalePrice thấp bất thường) → 1460 → 1458 dòng

2. Missing values:
   - Điền "None": PoolQC, MiscFeature, Alley, Fence, MasVnrType, FireplaceQu,
     GarageType/Finish/Qual/Cond, BsmtExposure/FinType1/FinType2/Qual/Cond
   - Điền 0: MasVnrArea
   - Điền median: LotFrontage
   - Điền theo cột khác: GarageYrBlt (= YearBuilt)
   - Điền mode: Electrical

3. Encode:
   - Ordinal (Po<Fa<TA<Gd<Ex → 0-5): ExterQual/Cond, BsmtQual/Cond, HeatingQC,
     KitchenQual, FireplaceQu, GarageQual/Cond, PoolQC
   - Nominal (one-hot, drop_first=True): 33 cột còn lại → 81 cột gốc thành 230 cột

4. Tách X/Y: X = 229 features, Y = SalePrice

5. Train/test split: 80/20, random_state=42

6. Scaling: StandardScaler (không dùng MinMax vì nhạy outlier hơn)

7. Cần lưu cho Phase 2: scaler.pkl, feature_columns.pkl 
<b>